# Intro

* What "dropout" means, how it works, and why it works (speculative)
* How to implement dropout in PyTorch, including switching it on and off

Dropout assigns probability of dropping, i.e. of node being dropped.

Effects:

1. Prevents a single node from learning too much
2. Forces the model to have distributed representations
3. Makes the model less reliant on individual nodes and thus more stable

Other observations

* Generally requires more training
* Can decrease training accuracy but increase generalization
* USually works better on deep than shallow networks

Gigantic and diverse training data already reduce the risk of overfitting during pretraining, so dropout is not so crucial

More useful during fine-tuning and instruction-tuning, because datasets tend to be smaller and more specific

# Code

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Using nn.Dropout

In [2]:
# define a dropout instance and make some data
prob = .2

dropout = nn.Dropout(p=prob)
x = torch.ones(100)
print('Initial data vector:\n ', x, '\n')

# let's see what dropout returns
y = dropout(x)
print('"Raw" post-dropout vector:\n ', y, '\n')

# normalized
print('Normed post-dropout vector:\n ', y*(1-prob), '\n')

# how many were dropped
print(f'{100*torch.sum(y==0)/len(x):.1f}% dropped (expected {prob*100:.1f}% )')

Initial data vector:
  tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]) 

"Raw" post-dropout vector:
  tensor([0.0000, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 0.0000, 0.0000,
        1.2500, 1.2500, 1.2500, 0.0000, 0.0000, 1.2500, 1.2500, 1.2500, 1.2500,
        0.0000, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500,
        0.0000, 0.0000, 1.2500, 1.2500, 1.2500, 0.0000, 1.2500, 1.2500, 1.2500,
        1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 0.0000, 1.2500,
        1.2500, 1.2500, 1.2500, 0.0000, 1.2500, 0.0000, 1.2500, 0.0000, 1.2500,
        1.2500, 1

## Using F.dropout

In [3]:
F.dropout(x, p = prob)

tensor([1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 0.0000,
        1.2500, 0.0000, 1.2500, 1.2500, 0.0000, 1.2500, 0.0000, 1.2500, 0.0000,
        1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 0.0000, 1.2500, 0.0000, 1.2500,
        1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500,
        0.0000, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 0.0000, 1.2500,
        1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500,
        1.2500, 0.0000, 1.2500, 1.2500, 1.2500, 0.0000, 1.2500, 1.2500, 0.0000,
        1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 0.0000, 0.0000, 1.2500, 0.0000,
        1.2500, 1.2500, 1.2500, 0.0000, 0.0000, 1.2500, 1.2500, 1.2500, 1.2500,
        0.0000, 1.2500, 1.2500, 1.2500, 0.0000, 1.2500, 1.2500, 1.2500, 0.0000,
        1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 0.0000, 0.0000, 1.2500, 1.2500,
        1.2500])

## Toggling Dropout

nn.Dropout reacts to eval and doesn't work

In [4]:
dropout.eval()
# model.eval() - usually
y = dropout(x)
print(y)

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])


When evaluation is switched on dropout is switched off, we do dropout only while training model.

F.dropout doesn't react to eval and still works

In [5]:
dropout.eval()
y = F.dropout(x)
print(y)

tensor([0., 2., 2., 0., 0., 2., 0., 2., 2., 2., 0., 0., 0., 0., 2., 0., 0., 2.,
        0., 2., 0., 2., 0., 2., 0., 2., 0., 2., 0., 2., 2., 0., 0., 2., 2., 2.,
        2., 2., 2., 0., 0., 0., 2., 0., 0., 0., 2., 0., 2., 2., 2., 2., 0., 2.,
        2., 0., 2., 2., 0., 2., 2., 0., 2., 2., 2., 2., 0., 0., 0., 0., 0., 0.,
        2., 2., 0., 0., 0., 2., 0., 2., 2., 0., 2., 0., 0., 2., 2., 2., 2., 2.,
        2., 2., 0., 0., 2., 0., 2., 0., 2., 2.])


we need to switch off manually

In [6]:
y = F.dropout(x, training=False)
print(y)

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])


The model needs to be reset after toggling into eval mode

In [12]:
dropout.train()
y = dropout(x)
print('train() mode on:\n', y*(1-prob), '\n') # with dropout

dropout.eval()
y = dropout(x)
print('eval() mode on:\n', y, '\n') # without dropout

# dropout.train()
y = dropout(x)
print('eval() remains on:\n', y) # still w/o dropout

train() mode on:
 tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 1., 1., 1., 1., 1.,
        0., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1.,
        1., 1., 0., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 0.,
        0., 0., 1., 1., 1., 1., 1., 1., 1., 1.]) 

eval() mode on:
 tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]) 

eval() remains on:
 tensor([1., 1., 1., 1., 1., 1., 1., 1., 1.,

# Conclusion

* Dropout invlolves "switching off" units during training, and is thought to promote distributed representations
* LLM pretraining sets are so large and diverse that dropout is less important compared to, e.g., CNNs
* Dropout is more useful in fine-tuning when datasets are smaller and overfitting risk is higher